In [1]:
import sys; sys.path.append('../3rdparty/ElasticKnots/3rdparty/ElasticRods/python')
import sys; sys.path.append('../3rdparty/ElasticKnots/python')
import elastic_rods, elastic_knots
import numpy as np, matplotlib.pyplot as plt, time, io, os
from scipy.sparse import coo_matrix
from scipy.sparse.linalg import eigsh
from scipy.linalg import eigh

from helpers import *
from parametric_curves import *
import py_newton_optimizer 

from linkage_vis import LinkageViewer as Viewer, CenterlineViewer
from tri_mesh_viewer import PointCloudViewer, PointCloudMesh

%load_ext autoreload
%autoreload 2

import parallelism
parallelism.set_max_num_tbb_threads(1)

from MEP import MEP

Failed to load offscreen viewer: Could not load compiled module; is OffscreenRenderer missing a dependency?


In [2]:
knot_name = '5_2/0001.obj'
file = '../data/L400-r0.2-UpTo9Crossings/' + knot_name
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 1, [rod_radius, rod_radius])
centerline = read_nodes_from_file(file)  # supported formats: obj, txt
pr = define_periodic_rod(centerline, material)
rod_list = elastic_knots.PeriodicRodList([pr])
len(rod_list.getDoFs())

1601

In [3]:
view = Viewer(rod_list, width=1024, height=800)
view.show()


/home/simon/miniconda3/envs/ElasticKnots/lib/python3.9/site-packages/jupyter_client/session.py:721: UserWarning: Message serialization failed with:
Out of range float values are not JSON compliant
Supporting this message is deprecated in jupyter-client 7, please make sure your message is JSON-compliant
  content = self.pack(content)


Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [4]:
def callback(problem, iteration):
    if iteration % 5 == 0:
        view.update()
for i in range(1,100):
    print(f"iterration: {i}")
    optimizerOptions = py_newton_optimizer.NewtonOptimizerOptions()
    optimizerOptions.niter = 1000
    optimizerOptions.gradTol = 1e-8
    hessianShift = 1e-4 * compute_min_eigenval_straight_rod(pr)

    problemOptions = elastic_knots.ContactProblemOptions()
    problemOptions.contactStiffness = 1e+3
    problemOptions.dHat = 2*rod_radius * 0.1*i
    fixedVars = []   
    
    report = elastic_knots.compute_equilibrium(
        rod_list, problemOptions, optimizerOptions, 
        fixedVars=fixedVars,
        externalForces=np.zeros(rod_list.numDoF()),
        softConstraints=[],
        callback=callback,
        hessianShift=hessianShift
        )
    view.update()

iterration: 1
0	0.909793	0.0220395	0.0220395	0.0800781	1
1	0.905868	0.0221897	0.0221897	0.0234375	1
2	0.904773	0.0218692	0.0218692	0.00195312	1
3	0.904658	0.0233784	0.0233784	1	1
4	0.9044	0.00556405	0.00556405	0.5	1
5	0.90432	0.0112283	0.0112283	1	1
6	0.904139	0.00324372	0.00324372	1	1
7	0.903996	0.0028148	0.0028148	1	1
8	0.903922	0.00773061	0.00773061	1	1
9	0.903792	0.00663966	0.00663966	0.09375	1
10	0.903771	0.0165499	0.0165499	0.5	1
11	0.903751	0.0115585	0.0115585	1	1
12	0.903696	0.00260237	0.00260237	1	1
13	0.90365	0.0281766	0.0281766	0.5	1
14	0.903566	0.0150096	0.0150096	0.25	1
15	0.903541	0.0216451	0.0216451	0.5	1
16	0.90346	0.0103894	0.0103894	1	1
17	0.903459	0.0330743	0.0330743	1	1
18	0.903395	0.0142802	0.0142802	1	1
19	0.903355	0.0121193	0.0121193	1	1
20	0.903318	0.00722174	0.00722174	1	1
21	0.903292	0.00170235	0.00170235	1	1
22	0.903253	0.00090272	0.00090272	1	1
23	0.903192	0.000822707	0.000822707	1	1
24	0.903097	0.000824323	0.000824323	1	1
25	0.902959	0.000942699	0.000942699

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	24.3941	521.051	521.051	1	1
1	4.83635	154.258	154.258	1	1
2	1.59051	45.5903	45.5903	1	1
3	1.04707	13.4824	13.4824	1	1
4	0.943246	3.5522	3.5522	1	1
5	0.926816	0.953053	0.953053	1	1
6	0.923664	0.267582	0.267582	1	1
7	0.922896	0.0765877	0.0765877	1	1
8	0.922582	0.0237568	0.0237568	1	1
9	0.922423	0.00993686	0.00993686	1	1
10	0.922317	0.00533105	0.00533105	1	1
11	0.922244	0.00495397	0.00495397	1	1
12	0.922192	0.00482303	0.00482303	1	1
13	0.922152	0.00421436	0.00421436	1	1
14	0.922122	0.00321866	0.00321866	1	1
15	0.922101	0.00226043	0.00226043	1	1
16	0.922085	0.00156466	0.00156466	1	1
17	0.922071	0.00114533	0.00114533	1	1
18	0.922058	0.000488806	0.000488806	1	1
19	0.922049	0.000292286	0.000292286	1	1
20	0.922045	0.0014715	0.0014715	1	1
21	0.922043	0.000244474	0.000244474	1	1
22	0.922042	0.000166901	0.000166901	1	1
23	0.922041	0.00857276	0.00857276	1	1
24	0.92204	0.00419533	0.00419533	0.125	0
25	0.922039	0.00399215	0.00399215	0.25	0
26	0.922038	0.00311431	0.00311431	0.125	0
27	0.922038	0.00

The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



0	24.8942	546.826	546.826	1	1
1	4.20971	142.629	142.629	1	1
2	1.40218	37.9831	37.9831	1	1
3	0.999923	10.1329	10.1329	1	1
4	0.939235	2.71002	2.71002	1	1
5	0.928195	0.741982	0.741982	1	1
6	0.925227	0.211812	0.211812	1	1
7	0.924093	0.0666241	0.0666241	1	1
8	0.923605	0.0249566	0.0249566	1	1
9	0.923364	0.012129	0.012129	1	1
10	0.923213	0.00698401	0.00698401	1	1
11	0.923104	0.00488826	0.00488826	1	1
12	0.923028	0.00422778	0.00422778	1	1
13	0.922972	0.00338256	0.00338256	1	1
14	0.922933	0.00256913	0.00256913	0.5	0
15	0.922926	0.089719	0.089719	0.015625	1
16	0.922922	0.0958993	0.0958993	0.00390625	1
17	0.922921	0.0963339	0.0963339	0.0078125	1
18	0.922921	0.0981404	0.0981404	1	1
19	0.922847	0.026048	0.026048	1	1
20	0.922837	0.00555987	0.00555987	1	1
21	0.922833	0.00102544	0.00102544	1	1
22	0.922829	0.000293019	0.000293019	1	1
23	0.922827	0.0005635	0.0005635	1	1
24	0.922825	0.000196705	0.000196705	0.5	0
25	0.922823	0.00795529	0.00795529	1	1
26	0.922822	0.00759865	0.00759865	0.125	1
27	0.922822	0

...

iterration: 97
Tau running away
||H||_2: -7.38582e+09
||M||_2: 0.125663
Scaled tau: -1.87072e+91
0	1.11979e+08	1.79337e+07	1.79337e+07	0	0


The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



iterration: 98
Tau running away
||H||_2: -7.94205e+09
||M||_2: 0.125663
Scaled tau: -2.0116e+91
0	1.21292e+08	1.91542e+07	1.91542e+07	0	0


The simulations will continue, but the result might be non-physical due to spurious contact forces between neighboring edges.
Consider decreasing the cross-section radius or using a coarser polyline.
Increasing the minContactEdgeDist parameter would remove non-physical contact forces, too, but only at the expense of topology preservation guarantees.



iterration: 99
Tau running away
||H||_2: -8.52888e+09
||M||_2: 0.125663
Scaled tau: -2.16024e+91
0	1.31191e+08	2.04358e+07	2.04358e+07	0	0


In [5]:
from helpers import write_obj
file = '../data/NoCollision/' + knot_name
write_obj(file, rod_list)

In [11]:
# Load the centerline from file...
file = '../data/NoCollision/' + knot_name
knot = read_nodes_from_file(file)
rod_radius = 0.2
material = elastic_rods.RodMaterial('ellipse', 2000, 0.3, [rod_radius, rod_radius])
pr = define_periodic_rod(knot[::4], material)
rod_list = elastic_knots.PeriodicRodList([pr])

In [12]:
view = Viewer(rod_list, width=1024, height=800)
view.show()

Renderer(camera=PerspectiveCamera(aspect=1.28, children=(PointLight(color='#999999', position=(0.0, 0.0, 5.0),…

In [13]:
from helpers import write_obj
file = '../data/NoCollision/reduced' + knot_name
write_obj(file, rod_list)